# Day 1: single-agent baselines on RoBERTa-base + SST-2

Two curves:
1. **MeZO** (zeroth-order, no backprop) — `lr=1e-6`, `eps=1e-3`, `model.eval()`, `torch.no_grad()` throughout.
2. **AdamW** (standard fine-tuning) — upper baseline.

Run the **smoke config** first (small train subset, few steps) to verify the pipeline doesn't crash. Then crank `STEPS` / `EPOCHS` for the real run.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForMaskedLM, AutoModelForSequenceClassification, AutoTokenizer

from src.data import get_sst2_loaders
from src.train import train_adamw, TrainHistory
from src.mezo import MeZOOptimizer
from src.prompt import (
    LABEL_WORDS, build_prompt_dataset, get_label_token_ids,
    prompt_loss_fn, prompt_evaluate,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
if DEVICE.type == 'cuda':
    print('gpu:   ', torch.cuda.get_device_name(0))

plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (9, 5)

device: cuda
gpu:    NVIDIA GeForce RTX 4060 Ti


## Config

Two presets — switch by changing `MODE`.

In [2]:
MODE = 'full'  # 'smoke' | 'preview' | 'full'

MODEL_NAME = 'roberta-base'
SEED = 0

if MODE == 'smoke':
    TRAIN_SUBSET = 512        # tiny shard
    MEZO_STEPS = 500          # ~minute on a 4060 Ti
    ADAMW_EPOCHS = 1
    EVAL_EVERY_MEZO = 100
    EVAL_EVERY_ADAMW = 100
elif MODE == 'preview':
    TRAIN_SUBSET = 1000       # match few-shot setup
    MEZO_STEPS = 5_000        # quick signal: is MeZO actually moving accuracy?
    ADAMW_EPOCHS = 3
    EVAL_EVERY_MEZO = 500
    EVAL_EVERY_ADAMW = 100
elif MODE == 'full':
    TRAIN_SUBSET = 1000       # MeZO paper uses 1k examples for few-shot fine-tuning on SST-2
    MEZO_STEPS = 20_000
    ADAMW_EPOCHS = 3
    EVAL_EVERY_MEZO = 500
    EVAL_EVERY_ADAMW = 200
else:
    raise ValueError(MODE)

BATCH_SIZE = 16
MAX_LENGTH = 128
MEZO_LR = 1e-6
MEZO_EPS = 1e-3
ADAMW_LR = 2e-5

torch.manual_seed(SEED)
print(f'MODE = {MODE}')

MODE = full


## Load data

In [3]:
raw = load_dataset("glue", "sst2")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
label_token_ids = get_label_token_ids(tokenizer)
MASK_TOKEN_ID = tokenizer.mask_token_id

print(f"mask_token_id  : {MASK_TOKEN_ID}")
print(f"label_token_ids: {label_token_ids}")
print(f"  0 = '{LABEL_WORDS[0]}' -> token {label_token_ids[0]} ('{tokenizer.decode([label_token_ids[0]])}')")
print(f"  1 = '{LABEL_WORDS[1]}' -> token {label_token_ids[1]} ('{tokenizer.decode([label_token_ids[1]])}')")

def make_prompt_loader(split_name: str, subset: int | None, batch_size: int, shuffle: bool):
    ds = raw[split_name]
    if subset:
        ds = ds.shuffle(seed=SEED).select(range(min(subset, len(ds))))
    enc = build_prompt_dataset(ds["sentence"], ds["label"], tokenizer, max_length=MAX_LENGTH)
    dataset = TensorDataset(enc["input_ids"], enc["attention_mask"], enc["labels"])
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=shuffle)

train_loader = make_prompt_loader("train",      TRAIN_SUBSET, BATCH_SIZE, shuffle=True)
val_loader   = make_prompt_loader("validation", None,         64,         shuffle=False)
print(f"train batches: {len(train_loader)},  val batches: {len(val_loader)}")

clf_loaders = get_sst2_loaders(MODEL_NAME, batch_size=BATCH_SIZE, max_length=MAX_LENGTH,
                                train_subset=TRAIN_SUBSET, seed=SEED)

mask_token_id  : 50264
label_token_ids: {0: 6587, 1: 372}
  0 = ' terrible' -> token 6587 (' terrible')
  1 = ' great' -> token 372 (' great')
train batches: 62,  val batches: 14


## Run 1 — MeZO baseline

Two key invariants enforced inside `train_mezo`:
- `model.eval()` (no dropout — would break the L(θ+εz) vs L(θ−εz) symmetry);
- `@torch.no_grad()` inside `MeZOOptimizer.step` (no autograd graph → memory stays at inference level).

In [4]:
from itertools import cycle
from tqdm import tqdm

torch.manual_seed(SEED)
mezo_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)
mezo_model.eval()

init_loss, init_acc = prompt_evaluate(mezo_model, val_loader, label_token_ids, MASK_TOKEN_ID, DEVICE)
print(f"init val loss={init_loss:.4f}  acc={init_acc:.4f}")

mezo_opt = MeZOOptimizer(mezo_model, lr=MEZO_LR, eps=MEZO_EPS)
mezo_hist = TrainHistory()
data_iter = cycle(train_loader)
log_every = max(1, EVAL_EVERY_MEZO // 10)
bar = tqdm(range(MEZO_STEPS), desc="MeZO")

for step in bar:
    ids, attn, labs = (t.to(DEVICE) for t in next(data_iter))
    batch = {"input_ids": ids, "attention_mask": attn, "labels": labs}

    def loss_fn():
        return prompt_loss_fn(mezo_model, batch, label_token_ids, MASK_TOKEN_ID)

    loss_plus = mezo_opt.step(loss_fn)

    if step % log_every == 0:
        mezo_hist.step.append(step)
        mezo_hist.train_loss.append(loss_plus)
        bar.set_postfix(loss=f"{loss_plus:.4f}")

    if (step + 1) % EVAL_EVERY_MEZO == 0 or step == MEZO_STEPS - 1:
        ev_loss, ev_acc = prompt_evaluate(mezo_model, val_loader, label_token_ids, MASK_TOKEN_ID, DEVICE)
        mezo_hist.eval_step.append(step + 1)
        mezo_hist.eval_loss.append(ev_loss)
        mezo_hist.eval_acc.append(ev_acc)
        bar.set_postfix(loss=f"{loss_plus:.4f}", val_acc=f"{ev_acc:.4f}")

print(f"final MeZO val acc: {mezo_hist.eval_acc[-1]:.4f}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

init val loss=0.4993  acc=0.7775


MeZO:   0%|          | 0/20000 [00:00<?, ?it/s]

MeZO:   0%|          | 0/20000 [00:00<?, ?it/s, loss=0.4603]

MeZO:   0%|          | 1/20000 [00:00<39:33,  8.43it/s, loss=0.4603]

...

MeZO: 100%|█████████▉| 19999/20000 [49:05<00:00,  7.36it/s, loss=0.0641]

MeZO: 100%|█████████▉| 19999/20000 [49:11<00:00,  7.36it/s, loss=0.1339, val_acc=0.9094]

MeZO: 100%|██████████| 20000/20000 [49:11<00:00,  1.88s/it, loss=0.1339, val_acc=0.9094]

MeZO: 100%|██████████| 20000/20000 [49:11<00:00,  6.78it/s, loss=0.1339, val_acc=0.9094]

final MeZO val acc: 0.9094


## Run 2 — AdamW upper baseline

In [5]:
torch.manual_seed(SEED)
adamw_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

adamw_hist = train_adamw(
    adamw_model, clf_loaders.train, clf_loaders.val, DEVICE,
    epochs=ADAMW_EPOCHS, lr=ADAMW_LR,
    eval_every=EVAL_EVERY_ADAMW, log_every=max(1, EVAL_EVERY_ADAMW // 10),
)
print(f'final AdamW val acc: {adamw_hist.eval_acc[-1]:.4f}')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


AdamW:   0%|          | 0/186 [00:00<?, ?it/s]

final AdamW val acc: 0.8922


## Comparison

MeZO needs **far more steps** than AdamW to reach a comparable accuracy — but **each step is backprop-free**, so memory stays at inference level. The X-axes are intentionally not normalized: this is wall-of-shame honest about MeZO's cost in iteration count.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(mezo_hist.eval_step, mezo_hist.eval_acc, marker='o', label='MeZO', color='royalblue')
ax.plot(adamw_hist.eval_step, adamw_hist.eval_acc, marker='s', label='AdamW', color='darkorange')
ax.axhline(init_acc, color='gray', linestyle=':', label=f'zero-shot ({init_acc:.2f})')
ax.set_xlabel('optimizer step')
ax.set_ylabel('SST-2 val accuracy')
ax.set_title('Validation accuracy vs. steps')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(mezo_hist.eval_step, mezo_hist.eval_loss, marker='o', label='MeZO', color='royalblue')
ax.plot(adamw_hist.eval_step, adamw_hist.eval_loss, marker='s', label='AdamW', color='darkorange')
ax.set_xlabel('optimizer step')
ax.set_ylabel('SST-2 val CE loss')
ax.set_title('Validation loss vs. steps')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'Day 1 baselines on RoBERTa-base + SST-2  (mode={MODE})', y=1.02)
plt.tight_layout()

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    'method': ['zero-shot (prompt)', 'MeZO', 'AdamW'],
    'final_val_acc': [init_acc, mezo_hist.eval_acc[-1], adamw_hist.eval_acc[-1]],
    'final_val_loss': [init_loss, mezo_hist.eval_loss[-1], adamw_hist.eval_loss[-1]],
    'optimizer_steps': [0, MEZO_STEPS, adamw_hist.eval_step[-1]],
})
summary

## Save results for later notebooks (Day 2 will overlay FedAvg-MeZO on top)

In [8]:
import json
out_dir = ROOT / 'outputs'
out_dir.mkdir(exist_ok=True)

def hist_to_dict(h):
    return {'step': h.step, 'train_loss': h.train_loss,
            'eval_step': h.eval_step, 'eval_loss': h.eval_loss, 'eval_acc': h.eval_acc}

with open(out_dir / f'day1_baselines_{MODE}.json', 'w') as f:
    json.dump({
        'mode': MODE,
        'model': MODEL_NAME,
        'init_val_acc': init_acc,
        'init_val_loss': init_loss,
        'mezo': hist_to_dict(mezo_hist),
        'adamw': hist_to_dict(adamw_hist),
        'config': {
            'train_subset': TRAIN_SUBSET, 'batch_size': BATCH_SIZE, 'max_length': MAX_LENGTH,
            'mezo_lr': MEZO_LR, 'mezo_eps': MEZO_EPS, 'mezo_steps': MEZO_STEPS,
            'adamw_lr': ADAMW_LR, 'adamw_epochs': ADAMW_EPOCHS,
        },
    }, f, indent=2)
print('saved →', out_dir / f'day1_baselines_{MODE}.json')

saved → C:\Users\olegk\Desktop\mezo\outputs\day1_baselines_full.json
